# N144 · 状态图、双向BFS与A*

**目标：** 把操作作为边并为启发式声明正确性条件。

**先修：** N081, N084, N112, N125。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 状态编码；最短步数；双向层扩展；admissible/consistent；过期状态；重开节点。

## 从问题到算法

操作序列题可以转成状态图：完整配置是顶点，一次合法操作是单位权边。双向搜索必须按整层扩展并维护已知路径上界与未扩展深度下界，而非随意在两队列第一次碰到时返回。A*把已走代价g与剩余代价下界h相加；目标首次入堆不代表最优，应在合法最小优先级弹出时终止。

## 最小实现

**本章接口：** `open_lock(deadends, target)`、`sliding_puzzle(board)`、`shortest_path_all_nodes(graph)`、`astar_puzzle(start, goal)`

In [1]:
from collections import deque
from heapq import heappush,heappop

def _bidirectional_distance(start,goal,neighbors,blocked=frozenset()):
    if start in blocked or goal in blocked: return -1
    if start==goal: return 0
    front=[{start},{goal}]; distance=[{start:0},{goal:0}]; depth=[0,0]; best=float('inf')
    while front[0] and front[1]:
        side=0 if len(front[0])<=len(front[1]) else 1; other=side^1; nxt=set()
        for u in front[side]:
            if u in distance[other]: best=min(best,distance[side][u]+distance[other][u])
            for v in neighbors(u):
                if v in blocked: continue
                value=distance[side][u]+1
                if v in distance[other]: best=min(best,value+distance[other][v])
                if v not in distance[side]: distance[side][v]=value; nxt.add(v)
        front[side]=nxt; depth[side]+=1
        if best<float('inf') and depth[0]+depth[1]>=best: return best
    return -1 if best==float('inf') else best

def open_lock(deadends,target):
    if len(target)!=4 or any(c not in '0123456789' for c in target): raise ValueError('four decimal digits required')
    def neighbors(s):
        for i,c in enumerate(s):
            for delta in (-1,1): yield s[:i]+str((int(c)+delta)%10)+s[i+1:]
    return _bidirectional_distance('0000',target,neighbors,set(deadends))

def _puzzle_neighbors(state,cols=3):
    zero=state.index(0); rows=len(state)//cols; r,c=divmod(zero,cols)
    for x,y in [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]:
        if 0<=x<rows and 0<=y<cols:
            pos=x*cols+y; out=list(state); out[zero],out[pos]=out[pos],out[zero]; yield tuple(out)

def sliding_puzzle(board):
    if len(board)!=2 or any(len(row)!=3 for row in board): raise ValueError('2x3 board required')
    start=tuple(x for row in board for x in row)
    if sorted(start)!=list(range(6)): raise ValueError('tiles must be 0..5 exactly once')
    return _bidirectional_distance(start,(1,2,3,4,5,0),_puzzle_neighbors)

def shortest_path_all_nodes(graph):
    n=len(graph)
    if n<=1: return 0
    full=(1<<n)-1; q=deque((i,1<<i,0) for i in range(n)); seen={(i,1<<i) for i in range(n)}
    while q:
        u,mask,steps=q.popleft()
        for v in graph[u]:
            new=mask|1<<v
            if new==full: return steps+1
            if (v,new) not in seen: seen.add((v,new)); q.append((v,new,steps+1))
    return -1

def astar_puzzle(start,goal):
    start,goal=tuple(start),tuple(goal)
    if len(start) not in (6,9) or sorted(start)!=list(range(len(start))) or sorted(goal)!=sorted(start): raise ValueError('matching 2x3 or 3x3 permutations required')
    positions={x:divmod(i,3) for i,x in enumerate(goal)}
    def heuristic(state):
        return sum(abs(i//3-positions[x][0])+abs(i%3-positions[x][1]) for i,x in enumerate(state) if x)
    distance={start:0}; heap=[(heuristic(start),0,start)]
    while heap:
        bound,g,u=heappop(heap)
        if g!=distance[u]: continue
        if u==goal: return g
        for v in _puzzle_neighbors(u):
            candidate=g+1
            if candidate<distance.get(v,float('inf')):
                distance[v]=candidate; heappush(heap,(candidate+heuristic(v),candidate,v))
    return -1

## 正确性、前提与复杂度

单位权BFS按距离层发现状态。双向算法维护两个完整前沿，所有尚未连接的更短路径若存在，就必须在当前已发现范围形成连接；当两前沿深度之和达到已有上界后可停止。曼哈顿距离不计空格，每次移动只把一个数字移动一格，因此不高估且满足一致性；A*仍保留更小g更新和过期项过滤。

**代价：** 搜索代价按可达状态图O(V+E)或带堆O((V+E)log V)衡量，不按棋盘格子数线性计算。全节点访问状态至多n·2^n；拼图最坏为阶乘状态。双向helper要求无向可逆操作；有向图反向侧必须使用反图。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

start=(1,2,3,4,0,5); goal=(1,2,3,4,5,0)
show_table(['状态','距目标最短步数'],[(s,astar_puzzle(s,goal)) for s in [start]+list(_puzzle_neighbors(start))])
print('锁0000到0009',open_lock([],'0009'))

状态,距目标最短步数
"(1, 2, 3, 4, 0, 5)",1
"(1, 0, 3, 4, 2, 5)",2
"(1, 2, 3, 4, 5, 0)",0
"(1, 2, 3, 0, 4, 5)",2


锁0000到0009 1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert open_lock(['0000'],'0000')==-1
assert open_lock([],'0000')==0 and open_lock([],'0009')==1
assert open_lock(['0201','0101','0102','1212','2002'],'0202')==6
assert sliding_puzzle([[1,2,3],[5,4,0]])==-1
assert shortest_path_all_nodes([[1,2,3],[0],[0],[0]])==4
assert shortest_path_all_nodes([[],[]])==-1
from itertools import permutations
# 一次完整反向BFS提供独立的全部2x3真值。
goal=(1,2,3,4,5,0); dist={goal:0}; q=deque([goal])
while q:
    u=q.popleft()
    for v in _puzzle_neighbors(u):
        if v not in dist: dist[v]=dist[u]+1; q.append(v)
for p in permutations(range(6)):
    expected=dist.get(p,-1)
    assert sliding_puzzle([list(p[:3]),list(p[3:])])==expected
    assert astar_puzzle(p,goal)==expected
from random import Random
rng=Random(144)
for _ in range(100):
    adj=[[] for _ in range(8)]
    for u in range(8):
        for v in range(u+1,8):
            if rng.randrange(4)==0: adj[u].append(v); adj[v].append(u)
    for s in range(8):
        distance={s:0}; q=deque([s])
        while q:
            u=q.popleft()
            for v in adj[u]:
                if v not in distance: distance[v]=distance[u]+1; q.append(v)
        for t in range(8): assert _bidirectional_distance(s,t,lambda u:adj[u])==distance.get(t,-1)
print('N144: 所有本章断言通过')

N144: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用位掩码表达已访问顶点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明不一致启发式需要谨慎重开节点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 752. Open the Lock（canonical）
- 773. Sliding Puzzle（canonical）
- 847. Shortest Path Visiting All Nodes（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。